Build Constituent Catalog metadata objects/containers

In [10]:
from astropy import units as astro_unit
import astropy.units as u
import astropy.units.cds as ucds
from astropy.table import *
from dataclasses import dataclass, field
from itertools import combinations
import lsdb
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hdps.catalogs import *

import cocat_column_meta


# .Adding to data classes for constituent reference catalogs 
This stuff goes in the part of hdps that builds AIC Refcat.  

I need to have metadata for each constituent catalog (cocat).  

This includes information for each cocat column

I need to (a) build a container for the column metadata, and (b) acquire and format the column metadata to put into the container.  

Each cocat will have a catalog container and a metadata container.  

## First, see what columns are available for Gaia and PanSTARRs HATS-ed versions provided at data.lsdb.io

In [5]:
# get the catalogs to see what columns are is available:
atlas_path = "/net/scratch/kmfas/atlas_refcat2/atlas"      # I HATS-ed this from the data that was already on Aurora (not all cols)
ps1_path = "/net/scratch/kmfas/panstarrs"                  # I downloaded this from data.lsdb.io
gaia_path = "/net/scratch/kmfas/gaia"                      # I downloaded this from data.lsdb.io
atlas = lsdb.open_catalog(atlas_path,columns="all")
ps = lsdb.open_catalog(ps1_path,columns="all")
gaia = lsdb.open_catalog(gaia_path,columns="all")

In [ ]:
stats = gaia.per_pixel_statistics(include_columns=["ra","dec"], include_stats=["row_count"])
stats.sort_values(["ra: row_count"], ascending=False).head(5)

In [ ]:
stats = atlas.per_pixel_statistics(include_columns=["ra","dec"], include_stats=["row_count"])
stats.sort_values(["ra: row_count"], ascending=False).head(5)

## Units!  each catalog gives its columns in different units.  I need to acquire and reconcile them to a common AIC unit system.


In [ ]:
@dataclass(frozen=True)
class UnitInfo:
    '''Units as listed in AIC Refcat, PanSTARRS Object, and Gaia Source Tables'''
    panstarrs_unit: str
    gaia_unit: str
    astropy_unit: astropy.units.core.Unit

AIC_UNITS: dict[str, UnitInfo] = {
#{"aic_unit","panstarrs_unit","gaia_unit"}
    'dimensionless': UnitInfo(panstarrs_unit='dimensionless', gaia_unit='', astropy_unit=u.dimensionless_unscaled),
    'ABMagnitude[mag]': UnitInfo(panstarrs_unit='AB magnitudes', gaia_unit='', astropy_unit=u.ABmag),
    'VegaMagnitude[mag]': UnitInfo(panstarrs_unit='', gaia_unit='mag', astropy_unit=u.mag(u.def_unit("Vega"))),
    'Angle[mas]': UnitInfo(panstarrs_unit='', gaia_unit='mas', astropy_unit=u.mas),
    'Angle[arcsec]': UnitInfo(panstarrs_unit='arcsec', gaia_unit='', astropy_unit=u.arcsec),
    'Angle[deg]': UnitInfo(panstarrs_unit='degrees', gaia_unit='deg', astropy_unit=u.deg),
    'AngularVelocity[mas/yr]': UnitInfo(panstarrs_unit='milliarcseconds per year', gaia_unit='mas.yr**-1', astropy_unit=u.mas / u.yr),
    'Time[MJD]': UnitInfo(panstarrs_unit='days', gaia_unit='', astropy_unit=ucds.MJD), # for panstarrs epochMean, which is in MJD 
    'Time[JD]': UnitInfo(panstarrs_unit='', gaia_unit='yr', astropy_unit=u.yr), # for gaia ref_epoch, which is in JD year
    'Time[day]': UnitInfo(panstarrs_unit='', gaia_unit='d', astropy_unit=u.d), # for gaia rv_time_duration, which is in days
    'Distance[pc]': UnitInfo(panstarrs_unit='', gaia_unit='pc', astropy_unit=u.pc),
    'Velocity[km/s]': UnitInfo(panstarrs_unit='', gaia_unit='km.s**-1', astropy_unit=u.km / u.s),
    'Metallicity[dex]': UnitInfo(panstarrs_unit='', gaia_unit='"dex"', astropy_unit=u.dex),
    'Flux[e-/s]': UnitInfo(panstarrs_unit='', gaia_unit='"electron".s**-1', astropy_unit=u.electron / u.s),
    'Temperature[K]': UnitInfo(panstarrs_unit='', gaia_unit='K', astropy_unit=u.K),
    'GravitySurface[log(cm/s/s)]': UnitInfo(panstarrs_unit='', gaia_unit='log(cm.s**-2)', astropy_unit=4.5 * u.dex(u.cm / u.s / u.s)),
    'Misc[1/um]': UnitInfo(panstarrs_unit='', gaia_unit='um**-1', astropy_unit=1 / u.micron),
    'Position[rad,rad]': UnitInfo(panstarrs_unit='', gaia_unit='[rad, rad]', astropy_unit=[u.rad,u.rad])
}

<font color="#e69138">Format constituent catalog column info info from Claude-generated CSV files (info pulled from the catalog table description sites).</font>

The following 2 cells really should only need to be run once. For PanSTARRS and Gaia, they do the following: 
- For each column in the catalog provided by data.lsdb.io, 
    - get the column name and define the name it would have in AIC
    - rename units to a uniform convention for AIC Refcat
    - add an is_default column; you'll have to manually go into the file to select which columns you want to load as default

The resulting files contain cocat column info that will go into ColumnInfo entries for each cocat.  
They will be called whenever you want to loop through columns to build ColumnInfo entries for Catalog Metadata classes.


In [ ]:
#--------------------------------------------------------------------
# PanSTARRS
# Only needs to be run once!!!!!!
#--------------------------------------------------------------------
# :: note that the HATS-ed version of PanSTARRS object table that lsdb provides is a combination of the ObjectThin and MeanObject tables (OTMO)
# :: so you need column information from both tables.

# Read in Claude-generated CSV files with tables of column names, units, and descriptions.
# These files will be left unchanged.  
# - read in the csv file for the meanobject table from: https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812411/PS1+MeanObjectView+table+fields 
ps1_meanobject_tabname = "./catalog_column_info/ps1_meanobject_cols.csv"
ps1_meanobject_cols = pd.read_csv(ps1_meanobject_tabname)
# - read in the csv file for the objectthin table from: https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812394/PS1+Database+object+and+detection+tables
ps1_objectthin_tabname = "./catalog_column_info/ps1_objectthin_cols.csv"
ps1_objectthin_cols = pd.read_csv(ps1_objectthin_tabname)

# Units: convert PS unit convention into ours (get list of unique PS OTMO units by calling np.unique(ps1_meanobject_cols['unit']) and np.unique(ps1_objectthin_cols['unit'])
# - once for the MeanObject table:
psmean_units_to_aic_units = [['AB magnitudes',    'arcsec',        'days',    'degrees',    'dimensionless'],
                             ['ABMagnitude[mag]', 'Angle[arcsec]', 'Time[MJD]', 'Angle[deg]', 'dimensionless']]
for unt in range(len(psmean_units_to_aic_units[0])):
    ps_unit = psmean_units_to_aic_units[0][unt]
    aic_unit = psmean_units_to_aic_units[1][unt]
    ps1_meanobject_cols.loc[ps1_meanobject_cols['unit']==ps_unit, 'unit'] = aic_unit
# - and again for the ObjectThin table:
psthin_units_to_aic_units = [['arcsec',        'days',       'degrees',    'dimensionless', 'milliarcseconds per year'],
                             ['Angle[arcsec]', 'Time[MJD]', 'Angle[deg]', 'dimensionless', 'AngularVelocity[mas/yr]']]
for unt in range(len(psthin_units_to_aic_units[0])):
    ps_unit = psthin_units_to_aic_units[0][unt]
    aic_unit = psthin_units_to_aic_units[1][unt]
    ps1_objectthin_cols.loc[ps1_objectthin_cols['unit']==ps_unit, 'unit'] = aic_unit

# Build dictionary with necessary info for each column 
ps1_object_cols_rows = []
# - loop through columns & build a dictionary (will be a row of a dataframe) for each with info that needs to be in ColumnInfo
for i in range(len(ps.columns)):
    # get column name (PS name and AIC name)
    col_name = ps.columns[i]
    col_aicname = "ps_"+col_name
    col_dtype = list(ps.dtypes)[i]
    # is it in ObjectThin or MeanObject table?
    col_mean = ps1_meanobject_cols['column_name']==col_name 
    col_thin = ps1_objectthin_cols['column_name']==col_name
    # get column unit
    if len(ps1_meanobject_cols[col_mean])>0:
        col_unit = ps1_meanobject_cols[col_mean].values[0][2]
        col_description = ps1_meanobject_cols[col_mean].values[0][3]
    elif len(ps1_objectthin_cols[col_thin])>0:
        col_unit = ps1_objectthin_cols[col_thin].values[0][2]
        col_description = ps1_objectthin_cols[col_thin].values[0][3]
    else:
        col_unit = ""
        col_description = ""
    # build a dictionary with the formatted info that needs to go into ColumnInfo entry, except for default status (set in another CSV file described below)
    row = {'column_name': col_name, 'column_aicname': col_aicname, 'dtype': col_dtype, 'unit': col_unit, 'description': col_description}
    ps1_object_cols_rows.append(row)
# - turn that list of dictionaries into a DataFrame to write to a new CSV file below
ps1_lsdb_object_cols = pd.DataFrame(ps1_object_cols_rows)

# Save to CSV
ps1_lsdb_object_cols.to_csv("./catalog_column_info/formatted_ps1_object_cols.csv",index=False)
# - if you need to re-do the defaults, uncomment this next line!  otherwise, DONT OVERWRITE THIS FILE!
#############ps1_lsdb_object_cols.to_csv("catalog_column_info/defaults_formatted_ps1_object_cols.csv",index=False)

In [8]:
#--------------------------------------------------------------------
# Gaia 
# Only needs to be run once!!!!!!
#--------------------------------------------------------------------

# Read in Claude-generated CSV file with table of column names, units, and descriptions.
# These files will be left unchanged. 
# - read in the csv file with info from: https://gaia.aip.de/metadata/gaiadr3/gaia_source/
gaia_col_tabname = "catalog_column_info/gaia_source_cols.csv"
gaia_source_cols = pd.read_csv(gaia_col_tabname)

# Units: convert Gaia unit convention into ours (get list of unique Gaia Source units by calling print(np.unique(list(gaia_source_cols['unit']))))
gaia_units_to_aic_units = [['"dex"',            '"electron".s**-1', 'K',              '[rad, rad]',         'd',       'deg',        'km.s**-1',       'log(cm.s**-2)',               'mag',                'mas',        'mas.yr**-1',              'nan',           'pc',           'um**-1', 'yr'],
                           ['Metallicity[dex]', 'Flux[e-/s]',       'Temperature[K]', 'Position[rad, rad]', 'Time[d]', 'Angle[deg]', 'Velocity[km/s]', 'GravitySurface[log(cm/s/s)]', 'VegaMagnitude[mag]', 'Angle[mas]', 'AngularVelocity[mas/yr]', 'dimensionless', 'Distance[pc]', 'Misc[1/um]', 'Time[MJD]']]
for unt in range(len(gaia_units_to_aic_units[0])):
    gaia_unit = gaia_units_to_aic_units[0][unt]
    aic_unit = gaia_units_to_aic_units[1][unt]
    if gaia_unit == "nan":
        gaia_source_cols.loc[gaia_source_cols['unit'].isna(), "unit"] = aic_unit
    else:
        gaia_source_cols.loc[gaia_source_cols['unit']==gaia_unit, 'unit'] = aic_unit

# Build dictionary with necessary info for each column 
gaia_source_cols_rows = []
# - loop through columns & build a dictionary (will be a row of a dataframe) for each with info that needs to be in ColumnInfo
for i in range(len(gaia.columns)):
    # get column name (Gaia name and AIC name)
    col_name = gaia.columns[i]
    col_aicname = "gaia_"+col_name
    col_dtype = list(gaia.dtypes)[i]
    col = gaia_source_cols['column_name']==col_name
    # get column unit
    if len(gaia_source_cols[col])>0:
        col_unit = gaia_source_cols[col].values[0][2]
        if isinstance(col_unit,float): col_unit = ""
        col_description = gaia_source_cols[col].values[0][3]
    else:
        col_unit=""
        col_description=""
    # build a dictionary with the formatted info that needs to go into ColumnInfo entry, except for default status (set in another CSV file described below)
    row = {'column_name': col_name, 'column_aicname': col_aicname, 'dtype': col_dtype, 'unit': col_unit,'description': col_description}
    gaia_source_cols_rows.append(row)
# - turn that list of dictionaries into a DataFrame to write to a new CSV file below
gaia_lsdb_source_cols = pd.DataFrame(gaia_source_cols_rows)

# Save to CSV
gaia_lsdb_source_cols.to_csv("./catalog_column_info/formatted_gaia_source_cols.csv",index=False)
# - if you need to re-do the defaults, uncomment this next line!  otherwise, DONT OVERWRITE THIS FILE!
########gaia_lsdb_source_cols.to_csv("catalog_column_info/defaults_formatted_gaia_source_cols.csv",index=False)

You also need to build a CSV file to store defaults.  It should have the COCAT column name and a boolean, True if you want the column to be a default.  Go into the file and select default columns.  I don't write anything to the file from here, because it would overwrite hand-selected defaults that you make by editing the file.  All this next cell does is print out `<COCAT_COLNAME>`,False for each column of PS and then Gaia.  You can copy/paste that into a file, and there is where you'll hand-select defaults.

In [22]:
col_info['column_name'][:2]

0       objName
1    objNameHMS
Name: column_name, dtype: object

In [26]:
# PanSTARRS
col_file = "./catalog_column_info/formatted_ps1_object_cols.csv"
col_info = pd.read_csv(col_file)
for col in col_info['column_name']:
    print(f'"{col}",False')

"objName",False
"objNameHMS",False
"objAltName1",False
"objAltName2",False
"objAltName3",False
"objPopularName",False
"objID",False
"uniquePspsOBid",False
"ippObjID",False
"surveyID",False
"htmID",False
"zoneID",False
"tessID",False
"projectionID",False
"skyCellID",False
"randomID",False
"batchID",False
"dvoRegionID",False
"processingVersion",False
"objInfoFlag",False
"astrometryCorrectionFlag",False
"qualityFlag",False
"raStack",False
"decStack",False
"raStackErr",False
"decStackErr",False
"raMean",False
"decMean",False
"raMeanErr",False
"decMeanErr",False
"pmra",False
"pmdec",False
"pmraErr",False
"pmdecErr",False
"epochMean",False
"posMeanChisq",False
"cx",False
"cy",False
"cz",False
"lambda",False
"beta",False
"l",False
"b",False
"nStackObjectRows",False
"nStackDetections",False
"nDetections",False
"ng",False
"nr",False
"ni",False
"nz",False
"ny",False
"gQfPerfect",False
"gMeanPSFMag",False
"gMeanPSFMagErr",False
"gMeanPSFMagStd",False
"gMeanPSFMagNpt",False
"gMeanPSFMagMin",False


## ..Create dict for each row for each table made available through lsdb
These will go into the catalog metadata structure, which will be referenced when building AIC Refcat.

The Background:
- AIC Refcat is built from multiple constituent catalogs
- The constituent catalogs are available through data.lsdb.io
- When loading the catalogs, we will want to know all the columns that are available to load and what information is available about them (dtype, unit, descriptor), and identify default columns to call (necessary for what we want to put in AIC Refcat).  
- To get this info about each available column, I had claude build a table for each catalog using the catalog column description webpages.  These tables include (a) column name, (b) dtype (don't use, instead use what lsdb has), (c) unit, and (d) a description of the column.  
- These tables must be reformatted becasue (a) the units are all over the place, I need them to all have the same naming convention, (b) I want to build a file that has a column with the default status of each colname (must go in manually to set these), and (c) the PanSTARRs object table available on data.ldsb.io is a combination of the MeanObject and ObjectThin tables, so the appropriate columns must be pulled from both. 

<font color="#e69138">Print out entries for ColumnInfo instances for each column  (cmd+/ to comment/uncomment highlighted blocks)</font>

In [ ]:
# #--------------------------------------------------------------------
# # PANSTARRS
# #--------------------------------------------------------------------
# # - read in formatted column info table
# ps1_object_cols = pd.read_csv("catalog_column_info/defaults_formatted_ps1_object_cols.csv")

# ps1_default_columns = ['objID', 'objInfoFlag', 'qualityFlag', 
#                        'raMean', 'decMean', 'raMeanErr', 'decMeanErr', 
#                        'pmra', 'pmdec', 'pmraErr', 'pmdecErr', 'epochMean', 
#                        'gMeanApMag', 'gMeanApMagErr', 'gMeanApMagStd', 'gMeanApMagNpt', 'gFlags', 
#                        'rMeanApMag', 'rMeanApMagErr', 'rMeanApMagStd', 'rMeanApMagNpt', 'rFlags', 
#                        'iMeanApMag', 'iMeanApMagErr', 'iMeanApMagStd', 'iMeanApMagNpt', 'iFlags']

# # - loop through ps1 columns, print ColumnInfo entries for column metadata 
# print("PANSTARRS_COLUMNS: dict[str, ColumnInfo] = {")
# for col in range(len(ps1_object_cols)):
#     col_info = ps1_object_cols.loc[col]
#     is_default = False
#     if col_info['column_name'] in ps1_default_columns: is_default = True
#     print(f'    "{col_info['column_name']}": ColumnInfo("{col_info['column_aicname']}", dtype="{col_info['dtype']}", unit="{col_info['unit']}", is_default={is_default}, description="{col_info['description']}"),')
# print("}")

#--------------------------------------------------------------------
# Gaia
#--------------------------------------------------------------------
# - read in formatted column info table with defaults set
gaia_source_cols = pd.read_csv("catalog_column_info/gaiadr3_source_colums_aicunits.csv") #defaults_formatted_gaia_source_cols.csv")

# if you're reading in from the CSV file that has "aicunits" in the name, you need to add a column with aicname.
gaia_source_cols['column_aicname'] = ["gaia_"+i for i in gaia_source_cols['column_name']]

gaia_default_columns = ['source_id', 'ref_epoch', 'ra', 'ra_error', 'dec', 'dec_error', 
                        'parallax', 'parallax_error', 'pmra', 'pmra_error', 'pmdec', 'pmdec_error', 
                        'phot_g_mean_flux', 'phot_g_mean_flux_error', 'phot_g_mean_mag', 
                        'phot_bp_mean_flux', 'phot_bp_mean_flux_error', 'phot_bp_mean_mag', 
                        'phot_rp_mean_flux', 'phot_rp_mean_flux_error', 'phot_rp_mean_mag', 
                        'phot_variable_flag', 'in_qso_candidates', 'in_galaxy_candidates', 'non_single_star', 'has_xp_continuous', 
                        'classprob_dsc_combmod_quasar', 'classprob_dsc_combmod_galaxy', 'classprob_dsc_combmod_star', 
                        'azero_gspphot', 'azero_gspphot_lower', 'azero_gspphot_upper', 
                        'ag_gspphot', 'ag_gspphot_lower', 'ag_gspphot_upper', 
                        'ebpminrp_gspphot', 'ebpminrp_gspphot_lower', 'ebpminrp_gspphot_upper']

# - loop through gaia cols, print ColumnInfo entries for column metadata 
print("GAIA_COLUMNS: dict[str, ColumnInfo] = {")
for col in range(len(gaia_source_cols)):
    col_info = gaia_source_cols.loc[col]
    is_default = False
    if col_info['column_name'] in gaia_default_columns: is_default = True
    print(f'    "{col_info['column_name']}": ColumnInfo("{col_info['column_aicname']}", dtype="{col_info['dtype']}", unit="{col_info['unit']}", is_default={is_default}, description="{col_info['description']}"),')
print("}")

<font color="#e69138">Whatever prints out from the above cell can be copy/pasted into the cocat_column_info.py module.  `PANSTARRS_COLUMNS` and `GAIA_COLUMNS` are each a dictionary, where each entry is `"<cocat_column_name>": ColumnInfo` for a single column.  The next section has examples for how these can be called and used.</font>

# Using the column metadata dictionaries

The `ColumnInfo` dataclass holds, for a single cocat column, all the information about that column.

The cocat column metadata dictionaries contain an entry for each cocat column, consisting of `<cocat_column_name>: ColumnInfo` 

Each cocat column metadata dictionary is called `<COCAT_NAME>_COLUMNS`.

These column dictionaries are called by cocat data containers, each of which is a dataclass called `<Cocat_Name>()`


<font color="#e69138">Using the dictionaries alone:</font>

In [36]:
from cocat_column_meta import GAIA_COLUMNS,PANSTARRS_COLUMNS,ATLAS_COLUMNS # import the cocat ColumnInfo metadata classes

In [37]:
type(GAIA_COLUMNS)

dict

In [34]:
GAIA_COLUMNS['solution_id']

ColumnInfo(aic_colname='gaia_solution_id', dtype='int64[pyarrow]', unit='dimensionless', is_default=False, description='Solution Identifier')

In [ ]:
# to print out keys and items for a column dictionary
print(GAIA_COLUMNS.keys())
print(GAIA_COLUMNS.items())

# to print out only default columns in a column dictionary
print(list([key for key, value in PANSTARRS_COLUMNS.items() if value.is_default]))

dict_keys(['solution_id', 'designation', 'source_id', 'random_index', 'ref_epoch', 'ra', 'ra_error', 'dec', 'dec_error', 'parallax', 'parallax_error', 'parallax_over_error', 'pm', 'pmra', 'pmra_error', 'pmdec', 'pmdec_error', 'ra_dec_corr', 'ra_parallax_corr', 'ra_pmra_corr', 'ra_pmdec_corr', 'dec_parallax_corr', 'dec_pmra_corr', 'dec_pmdec_corr', 'parallax_pmra_corr', 'parallax_pmdec_corr', 'pmra_pmdec_corr', 'astrometric_n_obs_al', 'astrometric_n_obs_ac', 'astrometric_n_good_obs_al', 'astrometric_n_bad_obs_al', 'astrometric_gof_al', 'astrometric_chi2_al', 'astrometric_excess_noise', 'astrometric_excess_noise_sig', 'astrometric_params_solved', 'astrometric_primary_flag', 'nu_eff_used_in_astrometry', 'pseudocolour', 'pseudocolour_error', 'ra_pseudocolour_corr', 'dec_pseudocolour_corr', 'parallax_pseudocolour_corr', 'pmra_pseudocolour_corr', 'pmdec_pseudocolour_corr', 'astrometric_matched_transits', 'visibility_periods_used', 'astrometric_sigma5d_max', 'matched_transits', 'new_matched_t

Using the column metadata classes with a cocat data container:

In [ ]:
# This is how to use the column dictionary <CATNAME>_COLUMNS in a cocat data container 

@dataclass
class TestCocatMeta:
    """
    Metadata container for a test cocat.

    This will be the basic format for the GaiaSourceMeta(), PanSTARRSMeta(), ATLASRefcat2Meta() cocat metadata containers.  
    It exists to demonstrate how the <COCAT_COLUMS> dictionaries will be used.
    This one uses GAIA_COLUMNS as the example.  

    This class holds cocat metadata, including:
    - columns
    - path to HATS-ed catalog 
    """
    cat_name: str = "testcocat"
    cat_fullname: str = "Test Cocat"
    col_map: dict[str, ColumnInfo] = field(default_factory=dict) # 
    path: str

    @classmethod # rename to build_colmap?
    def from_defaults(cls,extra_col_names: list[str] | None = None) -> "TestCatMeta":
        '''Create class instance with default columns, and additional columns if specified'''
        merged = {name: spec for name, spec in GAIA_COLUMNS.items() if spec.is_default}
        if extra_col_names:
            for name in extra_col_names:
                merged[name] = GAIA_COLUMNS[name]
        return cls(col_map=merged)


    def columns_to_open(self) -> list[str]:
        '''Return a list of original column names to call when opening/loading catalog with lsdb.open_catalog()'''
        return [name for name, spec in self.col_map.items()]# if not spec.is_tile]

    

@dataclass
class TestCocat:
    """Data + Metadata container for a test cocat
    
    This will be the basic format for the GaiaSource(), PanSTARRS(), and ATLASRefcat2() cocat containers.
    It exists to demonstrate how the CocatMeta() class will be used.  
    """
    metadata: TestCocatMeta
    data: lsdb.Catalog

    def open() -> "TestCocat":
        """Lazily load catalog with LSDB"""



In [ ]:
tcat_empty = TestCat() # calling it alone will not load anything.  
tcat_empty.col_map

TestCat(col_map={})

In [9]:
# TestCat.from_defaults() returns a class instance with a column map col_map with only default and added columns selected
# note that you can specify a default column (like "ra") and it'll still be loaded, no problem
# and also you can specify non-default columns (like "logg_gspphot") in here.
cat = TestCat.from_defaults(['ra','logg_gspphot']) 
cat.col_map

{'source_id': ColumnInfo(aic_colname='gaia_source_id', dtype='int64[pyarrow]', unit='dimensionless', is_default=True, description='Unique source identifier (unique within a particular Data Release)'),
 'ref_epoch': ColumnInfo(aic_colname='gaia_ref_epoch', dtype='double[pyarrow]', unit='Time[yr]', is_default=True, description='Reference epoch'),
 'ra': ColumnInfo(aic_colname='gaia_ra', dtype='double[pyarrow]', unit='Angle[deg]', is_default=True, description='Right ascension'),
 'ra_error': ColumnInfo(aic_colname='gaia_ra_error', dtype='float[pyarrow]', unit='Angle[mas]', is_default=True, description='Standard error of right ascension'),
 'dec': ColumnInfo(aic_colname='gaia_dec', dtype='double[pyarrow]', unit='Angle[deg]', is_default=True, description='Declination'),
 'dec_error': ColumnInfo(aic_colname='gaia_dec_error', dtype='float[pyarrow]', unit='Angle[mas]', is_default=True, description='Standard error of declination'),
 'parallax': ColumnInfo(aic_colname='gaia_parallax', dtype='dou

In [89]:
cat.columns_to_open()

NameError: name 'cat' is not defined

In [90]:
from cocats import ATLASRefcat2


In [94]:
atlas = ATLASRefcat2(path="/net/scratch/kmfas")
vars(atlas)

{'path': '/net/scratch/kmfas',
 'columns': ['Unnamed: 0',
  'ra',
  'dec',
  'parallax',
  'pmra',
  'pmdec',
  'g',
  'r',
  'i',
  'h32',
  'h64',
  'h256'],
 'filters': [['g', '<=', 19.0], ['dec', '>=', -20.0]],
 'catalog': None,
 'meta': None}

In [ ]:
@dataclass
class ATLASRefcat2:
    """Container for ATLAS-Refcat2 Catalog & metadata.

    Attributes
    ----------
    path (str) Full path to HATSed catalog (location on-disk or URL through data.lsdb.io)
    columns (list[str]) List of column names to load; input for lsdb.open_catalog() (expect elements [0,1,2] to be [ID,RA,Dec] column names)
    filters (list[tuple]) List of filters on rows; input for lsdb.open_catalog()
    catalog (lsdb.Catalog) Lazily-loaded catalog in HATs format; opened with lsdb.open_catalog()
    meta (pandas.DataFrame) meta information for catalog column names and dtypes; input for map_partitions()

    Example:
    --------
    >>> atlas = ATLASRefcat2(path="/path/to/atlas",columns=["id","ra","dec","g"],filters=[["dec",">",-20.0],["g","<",19.0]])
    """

    NAME = "atlas"
    FULL_NAME = "ATLAS_Refcat2"
    DEFAULT_PATH = "/net/scratch/kmfas/atlas_refcat2/atlas_0_01"

    def __init__(
        self,
        path: str,
        columns: list = None,
        filters: list = None,
        catalog: lsdb.Catalog = None,
        meta: pd.DataFrame = None,
    ):
        self.path = path
        if columns is not None:
            self.columns = columns
        else:
            self.columns = [
                "Unnamed: 0",
                "ra",
                "dec",
                "parallax",
                "pmra",
                "pmdec",
                "g",
                "r",
                "i",
                "h32",
                "h64",
                "h256",
            ]
        if filters is not None:
            self.filters = filters
        else:
            self.filters = [["g", "<=", 19.0], ["dec", ">=", -20.0]]
        self.catalog = catalog
        self.meta = meta

    @classmethod
    def open(cls,path):
        cat = lsdb.open_catalog(path)
        return cls(path=path,catalog=cat)

In [80]:
atlas = ATLASRefcat2.open("/net/scratch/kmfas/atlas_refcat2/atlas")
atlas.catalog

,ra,dec,parallax,pmra,pmdec,g,r,i,h32,h64,h256
npartitions=2181,,,,,,,,,,,
"Order: 2, Pixel: 0",double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],int64[pyarrow],int64[pyarrow]
"Order: 2, Pixel: 1",...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 766",...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 767",...,...,...,...,...,...,...,...,...,...,...


In [85]:
at = ATLASRefcat2(path="/net/scratch/kmfas/atlas_refcat2/atlas")
at_test = at.open(path="/net/scratch/kmfas/atlas_refcat2/atlas")
at_test.catalog

,ra,dec,parallax,pmra,pmdec,g,r,i,h32,h64,h256
npartitions=2181,,,,,,,,,,,
"Order: 2, Pixel: 0",double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],int64[pyarrow],int64[pyarrow]
"Order: 2, Pixel: 1",...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 766",...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 767",...,...,...,...,...,...,...,...,...,...,...


In [70]:
from aic_refcat import AICRefcat

In [73]:
AICRefcat.open("/net/scratch/kmfas/atlas_refcat2/atlas/")

RuntimeError: Necessary column tanseg_id not in AIC Refcat columns

In [101]:
ps = PanSTARRS("/net/scratch/kmfas/panstarrs")
ps.catalog = lsdb.open_catalog(ps.path)
ps.catalog

,decMean,decMeanErr,epochMean,gFlags,gMeanPSFMag,gMeanPSFMagErr,iFlags,iMeanPSFMag,iMeanPSFMagErr,nDetections,ng,ni,nr,ny,nz,objID,objInfoFlag,qualityFlag,raMean,raMeanErr,rFlags,rMeanPSFMag,rMeanPSFMagErr,surveyID,yFlags,yMeanPSFMag,yMeanPSFMagErr,zFlags,zMeanPSFMag,zMeanPSFMagErr
npartitions=9577,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 5, Pixel: 0",double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],double[pyarrow],double[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int64[pyarrow],int32[pyarrow],int16[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],double[pyarrow],double[pyarrow],int16[pyarrow],int32[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],double[pyarrow],double[pyarrow]
"Order: 5, Pixel: 1",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 12286",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 12287",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [95]:
@dataclass
class PanSTARRS:
    """
    Container for Pan-STARRS ObjectThin+MeanObject Table & metadata.

    Attributes
    ----------
    path (str) Full path to HATSed catalog (location on-disk or URL through data.lsdb.io)
    columns (list[str]) List of column names to load; input for lsdb.open_catalog() (expect elements [0,1,2] to be [ID,RA,Dec] column names)
    filters (list[tuple]) List of filters on rows; input for lsdb.open_catalog()
    catalog (lsdb.Catalog) Lazily-loaded catalog in HATs format; opened with lsdb.open_catalog()
    meta (pandas.DataFrame) meta information for catalog column names and dtypes; input for map_partitions()

    Example:
    --------
    >>> panstarrs = PanSTARRS(path="/path/to/panstarrs",columns=["objid","raMean","decMean","gMeanApMag"],filters=[["decMean",">",-20.0],["gMeanApMag","<",21.0]])

    """

    NAME = "panstarrs"
    FULL_NAME = "PanSTARRS_OTMO"
    DEFAULT_PATH = "s3://stpubdata/panstarrs/ps1/public/hats/otmo"

    def __init__(
        self,
        path: str,
        columns: list = None,
        filters: list = None,
        catalog: lsdb.Catalog = None,
        meta: pd.DataFrame = None,
    ):
        self.path = path
        if columns is not None:
            self.columns = columns
        else:
            self.columns = [
                "objID",
                "raMean",
                "decMean",
                "raMeanErr",
                "decMeanErr",
                "pmra",
                "pmdec",
                "pmraErr",
                "pmdecErr",
                "gMeanApMag",
                "gMeanApMagErr",
                "rMeanApMag",
                "rMeanApMagErr",
                "iMeanApMag",
                "iMeanApMagErr",
            ]
        if filters is not None:
            self.filters = filters
        else:
            self.filters = [
                ["gMeanApMag", ">", 19.0],
                ["gMeanApMag", "<=", 21.0],
                ["decMean", ">=", -20.0],
            ]
        # rename RA,Dec columns:
        if (
            (catalog is not None)
            and ("raMean" in catalog.columns)
            and ("decMean" in catalog.columns)
        ):
            catalog = catalog.rename(columns={"raMean": "ra", "decMean": "dec"})
        self.catalog = catalog
        self.meta = meta

In [97]:
PanSTARRS("/net/scratch/kmfas/panstarrs")

PanSTARRS()